# NLI TF-IDF Optimised (Direction 1)

Improvements over the baseline:
1. **LinearSVC convergence**:`max_iter=5000`
2. **Separate TF-IDF** for premise and hypothesis, plus diff/product features for directionality
3. **Feature engineering**: modals, quantifiers, double negation, character n-grams
4. **Removed erroneous baseline cells**; classifier is LinearSVC


## Dependencies
```bash
pip install numpy pandas scipy scikit-learn
```
Run from the project root `nlu_cwk/`, or ensure data are under `training_data/NLI/`.


In [41]:
# Install dependencies (run this cell first)
%pip install numpy pandas scipy scikit-learn



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Training

Fit TF-IDF + `LinearSVC`, then save `optimisation/nli_tfidf_artifacts/tfidf_linsvc.joblib` for demo use.


In [42]:
import re
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib


In [43]:
# Run from project root or from the optimisation/ folder
import os
_data_dir = "training_data/NLI" if os.path.exists("training_data/NLI/train.csv") else "../training_data/NLI"
if not os.path.exists(f"{_data_dir}/train.csv"):
    raise FileNotFoundError(f"Data not found. Run from the nlu_cwk/ project root. Tried: {_data_dir}")
train_df = pd.read_csv(f"{_data_dir}/train.csv")
dev_df = pd.read_csv(f"{_data_dir}/dev.csv")

print(train_df.head())
print(train_df.shape)
print(dev_df.shape)
print(train_df.columns)


                                             premise  \
0  yeah i don't know cut California in half or so...   
1                      actual names will not be used   
2          The film was directed by Randall Wallace.   
3   "How d'you know he'll sign me on?"Anse studie...   
4  In the light of the candles his cheeks looked ...   

                                          hypothesis  label  
0  Yeah. I'm not sure how to make that fit. Maybe...      1  
1  For the sake of privacy, actual names are not ...      1  
2  The film was directed by Randall Wallace and s...      1  
3       Anse looked at himself in a cracked mirror.       1  
4  Drew regarded his best friend and noted that i...      1  
(24432, 3)
(6736, 3)
Index(['premise', 'hypothesis', 'label'], dtype='object')


In [44]:
train_texts = train_df["premise"] + " [SEP] " + train_df["hypothesis"]
dev_texts = dev_df["premise"] + " [SEP] " + dev_df["hypothesis"]

y_train = train_df["label"]
y_dev = dev_df["label"]

print(train_texts.iloc[0])

yeah i don't know cut California in half or something [SEP] Yeah. I'm not sure how to make that fit. Maybe you could cut California in half or just resize it to make it fit.


In [45]:
NEG_WORDS = {"not", "no", "never", "nothing", "nobody", "none", "without", "neither", "nor", "n't"}
MODAL_WORDS = {"must", "might", "may", "can", "could", "would", "should", "will", "shall"}
QUANTIFIER_WORDS = {"all", "some", "most", "every", "any", "always", "never", "often", "sometimes"}

def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

def extract_pair_features(df):
    """Extended features: original 12 dims + modals + quantifiers + double negation + char n-gram overlap."""
    features = []

    for _, row in df.iterrows():
        p = str(row["premise"]).lower()
        h = str(row["hypothesis"]).lower()

        p_tokens = tokenize(p)
        h_tokens = tokenize(h)

        p_set = set(p_tokens)
        h_set = set(h_tokens)

        # === Original features ===
        len_p = len(p_tokens)
        len_h = len(h_tokens)
        len_diff = abs(len_p - len_h)

        overlap_count = len(p_set & h_set)
        overlap_ratio_h = overlap_count / len(h_set) if len(h_set) > 0 else 0.0
        overlap_ratio_p = overlap_count / len(p_set) if len(p_set) > 0 else 0.0
        jaccard = overlap_count / len(p_set | h_set) if len(p_set | h_set) > 0 else 0.0

        p_neg = int(any(w in NEG_WORDS for w in p_tokens))
        h_neg = int(any(w in NEG_WORDS for w in h_tokens))
        neg_mismatch = int(p_neg != h_neg)

        p_nums = set(re.findall(r"\d+", p))
        h_nums = set(re.findall(r"\d+", h))
        same_number = int(len(p_nums) > 0 and len(h_nums) > 0 and p_nums == h_nums)
        number_mismatch = int((len(p_nums) > 0 or len(h_nums) > 0) and p_nums != h_nums)

        # === Extra: modals (modality in hypothesis affects entailment) ===
        p_modal = int(any(w in MODAL_WORDS for w in p_tokens))
        h_modal = int(any(w in MODAL_WORDS for w in h_tokens))
        modal_mismatch = int(p_modal != h_modal)

        # === Extra: quantifiers (universal/existential) ===
        p_quant = int(any(w in QUANTIFIER_WORDS for w in p_tokens))
        h_quant = int(any(w in QUANTIFIER_WORDS for w in h_tokens))
        quant_mismatch = int(p_quant != h_quant)

        # === Extra: double negation (not ... not in p or h) ===
        double_neg_p = int(re.search(r"not\b.*\bnot\b|n't.*n't", p) is not None)
        double_neg_h = int(re.search(r"not\b.*\bnot\b|n't.*n't", h) is not None)

        # === Extra: char 3-gram overlap (morphological variation) ===
        def char_ngrams(s, n=3):
            s = re.sub(r"\s+", "", s)
            return set(s[i:i+n] for i in range(len(s)-n+1)) if len(s) >= n else set()
        c3_p, c3_h = char_ngrams(p, 3), char_ngrams(h, 3)
        char_overlap = len(c3_p & c3_h) / len(c3_p | c3_h) if (c3_p or c3_h) else 0.0

        features.append([
            len_p, len_h, len_diff,
            overlap_count, overlap_ratio_h, overlap_ratio_p, jaccard,
            p_neg, h_neg, neg_mismatch,
            same_number, number_mismatch,
            p_modal, h_modal, modal_mismatch,
            p_quant, h_quant, quant_mismatch,
            double_neg_p, double_neg_h,
            char_overlap
        ])

    return np.array(features, dtype=float)


In [46]:
X_train_extra = extract_pair_features(train_df)
X_dev_extra = extract_pair_features(dev_df)

print("Extra feature shape:", X_train_extra.shape)  # 21 dims (12 original + 9 extra)
print(X_train_extra[:3])


Extra feature shape: (24432, 21)
[[11.         25.         14.          7.          0.33333333  0.63636364
   0.28        0.          1.          1.          0.          0.
   0.          1.          1.          0.          0.          0.
   0.          0.          0.21212121]
 [ 6.         10.          4.          4.          0.4         0.66666667
   0.33333333  1.          1.          0.          0.          0.
   1.          0.          1.          0.          0.          0.
   0.          0.          0.24489796]
 [ 7.         11.          4.          7.          0.63636364  1.
   0.63636364  0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.63265306]]


In [47]:
# === Separate TF-IDF for premise/hypothesis + diff/product for directionality ===
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit on concatenated corpus for a consistent vocabulary
all_premise = pd.concat([train_df["premise"], dev_df["premise"]])
all_hypo = pd.concat([train_df["hypothesis"], dev_df["hypothesis"]])
tfidf.fit(all_premise.astype(str) + " " + all_hypo.astype(str))

X_train_p = tfidf.transform(train_df["premise"])
X_train_h = tfidf.transform(train_df["hypothesis"])
X_dev_p = tfidf.transform(dev_df["premise"])
X_dev_h = tfidf.transform(dev_df["hypothesis"])

# Directional features: difference and element-wise product (sparse)
X_train_diff = X_train_p - X_train_h
X_train_prod = X_train_p.multiply(X_train_h)
X_dev_diff = X_dev_p - X_dev_h
X_dev_prod = X_dev_p.multiply(X_dev_h)

# Stack: P | H | diff | product | hand-crafted features
X_train = hstack([
    X_train_p, X_train_h, X_train_diff, X_train_prod,
    csr_matrix(X_train_extra)
])
X_dev = hstack([
    X_dev_p, X_dev_h, X_dev_diff, X_dev_prod,
    csr_matrix(X_dev_extra)
])

print("Feature matrix shape:", X_train.shape)
print("Dev shape:", X_dev.shape)

# === LinearSVC: fix convergence (max_iter=5000) ===
clf = LinearSVC(C=1.0, max_iter=5000)
clf.fit(X_train, y_train)

# Persist model for Deliverable 2 demo (load without retraining)
_ARTIFACT_DIR = os.path.join("optimisation", "nli_tfidf_artifacts")
os.makedirs(_ARTIFACT_DIR, exist_ok=True)
_ARTIFACT_FILE = os.path.join(_ARTIFACT_DIR, "tfidf_linsvc.joblib")
joblib.dump({"tfidf": tfidf, "clf": clf}, _ARTIFACT_FILE)
print(f"Saved artifacts to {_ARTIFACT_FILE}")


Feature matrix shape: (24432, 428045)
Dev shape: (6736, 428045)
Saved artifacts to optimisation/nli_tfidf_artifacts/tfidf_linsvc.joblib


/opt/homebrew/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## 2. Evaluation (development set)

Metrics on `dev.csv` only.


In [48]:
dev_pred = clf.predict(X_dev)

print("=== Optimised NLI evaluation ===")
print("Accuracy:", accuracy_score(y_dev, dev_pred))
print("F1 (macro):", f1_score(y_dev, dev_pred, average="macro"))
print("F1 (binary):", f1_score(y_dev, dev_pred))
print(classification_report(y_dev, dev_pred, target_names=["not entailment", "entailment"]))


=== Optimised NLI evaluation ===
Accuracy: 0.6454869358669834
F1 (macro): 0.6448820714541363
F1 (binary): 0.6595380667236954
                precision    recall  f1-score   support

not entailment       0.64      0.62      0.63      3258
    entailment       0.65      0.67      0.66      3478

      accuracy                           0.65      6736
     macro avg       0.65      0.64      0.64      6736
  weighted avg       0.65      0.65      0.65      6736



## 3. Test inference (demo)

Optional: loads `tfidf_linsvc.joblib` if present. Requires `extract_pair_features` from above.


In [49]:
# === Test inference (test_data/NLI/test.csv) ===
_test_candidates = (
    "test_data/NLI/test.csv",
    "../test_data/NLI/test.csv",  # when running from optimisation/
)
test_path = next((p for p in _test_candidates if os.path.exists(p)), None)

# Optional: load artifacts from a previous training run (skip training cells)
_artifact_candidates = (
    os.path.join("optimisation", "nli_tfidf_artifacts", "tfidf_linsvc.joblib"),
    os.path.join("..", "optimisation", "nli_tfidf_artifacts", "tfidf_linsvc.joblib"),
)
for _p in _artifact_candidates:
    if os.path.exists(_p):
        _bundle = joblib.load(_p)
        tfidf = _bundle["tfidf"]
        clf = _bundle["clf"]
        print(f"Loaded model from {_p}")
        break

if test_path:
    print(f"Loading test set: {test_path}")
    test_df = pd.read_csv(test_path)
    X_test_extra = extract_pair_features(test_df)
    X_test_p = tfidf.transform(test_df["premise"])
    X_test_h = tfidf.transform(test_df["hypothesis"])
    X_test = hstack([X_test_p, X_test_h, X_test_p - X_test_h, X_test_p.multiply(X_test_h), csr_matrix(X_test_extra)])
    test_pred = clf.predict(X_test)
    pd.DataFrame({"label": test_pred}).to_csv("tf_IDF_predictions.csv", index=False)
    print("Saved tf_IDF_predictions.csv")
else:
    print("test_data/NLI/test.csv not found; skipping inference")


Loaded model from optimisation/nli_tfidf_artifacts/tfidf_linsvc.joblib
Loading test set: ../test_data/NLI/test.csv
Saved tf_IDF_predictions.csv
